In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import pandas as pd  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402
from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_contrastive_synchrony import (  # noqa: E402
    RESULTS_SUBDIR,
    STAGE,
    _build_arg_parser,
    run,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.models.config import DEFAULT_CONFIG_PATH, config_to_dict, load_config  # noqa: E402
from src.models.encoders import ENCODERS, n_parameters  # noqa: E402
from src.models.training import load_checkpoint  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 08 — Contrastive synchrony model

An encoder learns to embed a few seconds of a subject's wavelet power so that windows
from **different subjects at the same moment of the same track** land close together and
windows from different moments land apart. Group synchrony of a window is then the mean
pairwise cosine similarity of everyone's embeddings, judged against time-shifted
surrogates; windows above the null threshold are the candidate synchronisation intervals.

**The method is documented in
[`docs/analyses/08-contrastive-synchrony/`](../../docs/analyses/08-contrastive-synchrony/README.md)**
— this notebook does not restate it.

**What this notebook is for.** A lightweight look at the model and a cheap test of a
setting before a GPU job is spent on it. It calls `run()` from
[`scripts/run_contrastive_synchrony.py`](../../scripts/run_contrastive_synchrony.py) — the
same function [`train.pbs`](../../jobs/metacentrum/08-contrastive-synchrony/train.pbs)
calls — on a few subjects and minutes, then shows what it wrote. It reads the stage-03
wavelet store, memory-mapped, so only the subjects, channels and minutes it keeps are
read.

## Configuration

Every setting lives in
[`config/models/contrastive_synchrony.yaml`](../../config/models/contrastive_synchrony.yaml).
`CONFIG` may point at a variant file holding only the keys it changes; `OVERRIDES` change
single keys on top. The small-run overrides below only shrink the run — drop them to see
the configuration the job uses.

In [ ]:
CONFIG = DEFAULT_CONFIG_PATH
RUN_NAME = "notebook"

# Your experiment: any "section.key=value" from the config file.
OVERRIDES = [
    # "model.name=mlp",
    # "data.window_s=8",
]

# Small-run settings, so the cell below finishes in a minute or two on a CPU.
SMALL_RUN = [
    "training.n_folds=2",
    "training.epochs=5",
    "training.steps_per_epoch=30",
    "training.val_steps=5",
    "scoring.n_null=50",
]
N_SUBJECTS = 6
N_TIMES = 4500  # frames: 3 min at 25 Hz

WAVELET_ROOT = None  # None = data/processed; notebooks/cache for the strips
OUT_DIR = ProjectPaths.NOTEBOOKS_DIR / "08-contrastive-synchrony" / "out"

config = load_config(CONFIG, [*SMALL_RUN, *OVERRIDES, f"run_name={RUN_NAME}"])
print(yaml.safe_dump(config_to_dict(config), sort_keys=False))
print("Registered encoders:", sorted(ENCODERS))
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Run

Trains, scores every window of every track and writes the tables and figures under
`OUT_DIR`. The full-size pass is the job's.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--config",
        str(CONFIG),
        "--set",
        *SMALL_RUN,
        *OVERRIDES,
        "--run_name",
        RUN_NAME,
        "--n_subjects",
        str(N_SUBJECTS),
        "--n_times",
        str(N_TIMES),
        "--results_dir",
        str(OUT_DIR / "results"),
        "--save_dir",
        str(OUT_DIR / "plots"),
        *(["--wavelet_data_dir", str(WAVELET_ROOT)] if WAVELET_ROOT else []),
    ]
)
run(args)

## The model and what it found

One encoder per cross-fitting fold; fold 0's is printed. The training history has one
row per fold and epoch — held-out accuracy above chance is the evidence that the encoder
found shared, time-locked structure rather than memorising moments. Candidate intervals
are runs of windows whose group synchrony beats their block's null threshold; every
window was scored by the fold model that never saw it.

In [ ]:
RUN_DIR = OUT_DIR / "results" / RESULTS_SUBDIR / RUN_NAME
model = load_checkpoint(RUN_DIR / "model_fold0.pt", torch.device("cpu"))
print(model)
print(f"{n_parameters(model):,} trainable parameters")

display(pd.read_csv(RUN_DIR / "history.csv").round(3))
display(pd.read_csv(RUN_DIR / "candidates.csv").round(3))

## Figures

Drawn by [`src/visualization/contrastive_plots.py`](../../src/visualization/contrastive_plots.py),
which the job uses too — so what is shown here is what the job produces.

In [ ]:
PLOTS_DIR = ProjectPaths.analysis_plots_dir(STAGE, RUN_NAME, root=OUT_DIR / "plots")
for png in sorted(PLOTS_DIR.glob("*.png")):
    display(Markdown(f"**{png.name}**"))
    display(Image(filename=str(png)))

## The full run

The job's own output for the configured run name, when it exists — the figures to read
once a setting has been chosen here.

In [ ]:
FULL_RUN = "baseline"
full_plots = ProjectPaths.analysis_plots_dir(STAGE, FULL_RUN)
full_results = (
    ProjectPaths.PROJECT_ROOT / "results" / "psilo_music" / RESULTS_SUBDIR / FULL_RUN
)
if not full_plots.exists():
    print(
        f"No full run '{FULL_RUN}' yet: submit jobs/metacentrum/08-contrastive-synchrony/train.pbs."
    )
else:
    display(pd.read_csv(full_results / "candidates.csv").round(3))
    for png in sorted(full_plots.glob("*.png")):
        display(Markdown(f"**{png.name}**"))
        display(Image(filename=str(png)))